# Puck Lab: improve the neural predictor

One laptop per team. Open this notebook from student/ in the full package. Install training-requirements.txt before the event. No API key or ChatGPT Work subscription.

Diagnose the original network, train a candidate, and keep the planner and evaluator fixed. Submit your training choices, code, checkpoint and complete results. A physics-equation replacement does not count as the neural repair.


In [ ]:
from pathlib import Path
Path('results').mkdir(exist_ok=True)
import json
import numpy as np
import matplotlib.pyplot as plt
from lab import Dynamics, physics, rollout, candidates, predicted_paths, experiment, plot_shot

model = Dynamics()
start, target = (.16, .7), (.83, .72)
meta = json.loads(Path('../assets/metadata.json').read_text())
print(f"Training transitions: {meta['trainingSamples']:,}")
print(f"Validation one-step position RMSE: {100*meta['positionRMSE']:.3f}% of table width")
print('Target radius: 6.5% of table width. Position RMSE is NOT target success.')


## 1. Reproduce before explaining

The following action pools are nested: the 512-shot search contains the first 128 candidates. Same model, target, horizon and seed. Only the search budget changes. Predict which result should improve before running this cell.

In [ ]:
runs = {}
for budget in (32, 128, 512, 2048):
    r = experiment(model, start, target, budget=budget)
    runs[budget] = r
    print(f"{budget:4d} candidates | predicted miss {r['predicted_miss']:.3f} | actual miss {r['actual_miss']:.3f}")
fig = plot_shot(runs[512], start, target)
plt.show()


## 2. Separate one-step error from accumulated error

A one-step prediction below receives the **true current state**. The free rollout receives its own last prediction. Both use the same checkpoint. Does the divergence start immediately, grow gradually, or change sharply around contact? Inspect the trajectory rather than labeling every divergence as overfitting.

In [ ]:
r = runs[512]
truth, free = r['actual'], r['predicted']
one_step = model(truth[:-1])
one_error = np.linalg.norm(one_step[:,:2] - truth[1:,:2], axis=1)
free_error = np.linalg.norm(free[1:,:2] - truth[1:,:2], axis=1)
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(range(1,36), one_error, label='One step from true state')
ax.plot(range(1,36), free_error, label='Free rollout')
ax.set(xlabel='Step', ylabel='Position error (table widths)')
ax.legend(); fig.tight_layout(); plt.show()
print('Mean one-step error:', one_error.mean())
print('Final free-rollout error:', free_error[-1])


## 3. Does selection favour optimistic errors?

Evaluate the *same* candidate pool against the simulator after prediction. Simulator outcomes are diagnostic evidence only; they must not enter the planner's selection rule. Compare errors for all candidates with those for the candidates the learned model likes. This plot alone does not establish a universal causal effect.

In [ ]:
actions = candidates(512, max_speed=.11)
paths = predicted_paths(model, start, actions)
predicted_miss = np.linalg.norm(paths[:,-1,:2] - target, axis=1)
actual_miss = np.array([np.linalg.norm(rollout(start,a,35,physics)[-1,:2]-target) for a in actions])
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(predicted_miss, actual_miss, s=10, alpha=.35, color='#727987')
chosen = runs[512]['selected']
ax.scatter(predicted_miss[chosen],actual_miss[chosen],s=75,color='#245bd6',label='Planner choice')
limit = max(predicted_miss.max(),actual_miss.max())
ax.plot([0,limit],[0,limit],':',color='black')
ax.set(xlabel='Predicted miss',ylabel='Actual miss',title='Above diagonal = optimistic prediction')
ax.legend(); plt.show()
print('Mean optimism across pool:', (actual_miss-predicted_miss).mean())
print('Optimism of selected shot:', actual_miss[chosen]-predicted_miss[chosen])


## 4. Run one named neural experiment

Choose a new RUN_NAME for every trial so earlier checkpoints are preserved. First run the defaults as your trainable baseline. Then change one factor and set COMPARE_RUN to the first run name. You edit train_repair.py (layers/loss/data), neural_model.py (features/inference) or the settings below. Do not alter the planner or evaluator.

A plausible change can fail: extra collision loss may sacrifice free-motion accuracy; a larger network may need more data. Keep negative results. See EXPERIMENTS.md for concrete hypotheses and code locations.


In [ ]:
import subprocess, sys
RUN_NAME = "experiment-1"  # Choose a new name when rerunning with a new hypothesis.
COMPARE_RUN = None        # For example, "experiment-1" on your next run.
DATA = "original"
EPOCHS = 40
CONTACT_WEIGHT = 1
GEOMETRY = False
FREE_HEAD = False  # Advanced: independently learn the free-motion branch.
TRAINING_CONFIG = dict(data=DATA, epochs=EPOCHS, contact_weight=CONTACT_WEIGHT, geometry=GEOMETRY, free_head=FREE_HEAD)
command = [sys.executable, "run_experiment.py", "--name", RUN_NAME, "--data", DATA, "--epochs", str(EPOCHS), "--contact-weight", str(CONTACT_WEIGHT)]
if GEOMETRY:
    command.append("--geometry")
if FREE_HEAD:
    command.append("--free-head")
if COMPARE_RUN:
    command.extend(["--compare", COMPARE_RUN])
subprocess.run(command, check=True)
RUN_DIRECTORY = Path("results/runs") / RUN_NAME


## Training observations and portable inference

`training-observations.npz` contains the original 30,000 transitions. The starter splits whole 25-step trajectories, never adjacent rows at random. `neural-training.npz` is an optional larger dataset with independent training and validation samples, known geometry, stronger pushes and more near-boundary states. Neither dataset contains final evaluation scenes.

`train_repair.py` saves learned weights and a training report under results/. `neural_model.py` runs those weights using NumPy; `repair.py` loads the checkpoint. Inspect these three small files rather than the whole repository. Restart the kernel after editing Python inference code.


In [ ]:
from neural_model import NeuralDynamics
with np.load("../assets/training-observations.npz", allow_pickle=False) as data:
    training_states = data["states"]
repair = NeuralDynamics(RUN_DIRECTORY / "network.json")
probe = repair(training_states[:8])
assert probe.shape == (8,4) and np.isfinite(probe).all()
print("Loaded your actual trained checkpoint:", RUN_DIRECTORY / "network.json")


## 5. Inspect your model in the browser and submit

Open the workshop website. Under **Try your team’s trained model**, select `results/runs/<your-run>/network.json`. The website runs those exact weights locally. Choose a target, find a shot and execute it. The browser view is diagnostic; the score below covers every development scene.

Submit your chosen run folder and any edited Python files. It includes weights, training configuration and the fixed evaluation result. Add your hypothesis and interpretation below. Hosts score one frozen checkpoint on separate scenes.

The browser accepts the supplied starter’s export format, raw or geometry inputs, with hidden widths up to 256. Custom architectures beyond that interface require a matching inference adapter; use the Python evaluator and ask a host to review the adapter.


In [ ]:
score = json.loads((RUN_DIRECTORY / "score.json").read_text())
report = {
    "team": "EDIT",
    "hypothesis": "EDIT",
    "result_that_would_disprove_it": "EDIT",
    "observation": "EDIT",
    "interpretation": "EDIT",
    "training_configuration": TRAINING_CONFIG,
    "scene_results": score,
    "regressions_or_limitations": "EDIT",
}
(RUN_DIRECTORY / "team-report.json").write_text(json.dumps(report, indent=2))
print(f"{score['hits']}/{score['total']} targets. Add your evidence to", RUN_DIRECTORY / "team-report.json")
